In [4]:
import re
import json
import logging
import torch
import os
import numpy as np
from typing import List, Dict, Any, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from datasets import Dataset
from presidio_analyzer import PatternRecognizer, Pattern, RecognizerRegistry, AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpArtifacts
from presidio_analyzer.predefined_recognizers import TransformersRecognizer
from presidio_analyzer.nlp_engine import NlpEngine
from presidio_analyzer.predefined_recognizers import AzureAILanguageRecognizer

In [ ]:
import re
import os
import json
import logging
from typing import List, Dict, Any, Optional, Set

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset

# Import necessary Presidio modules
from presidio_analyzer import (
    RecognizerRegistry, 
    PatternRecognizer, 
    Pattern,
    AnalyzerEngine,
    EntityRecognizer  # This was missing
)
from presidio_analyzer.nlp_engine import NlpEngine, NlpArtifacts

# Define the missing generate_regex function
def generate_regex(entity_list):
    """
    Generate a regex pattern from a list of entities
    
    Args:
        entity_list (List[str]): List of entity strings
        
    Returns:
        str: Regex pattern matching any of the entities
    """
    if not entity_list:
        return r"\b(?:placeholder_that_wont_match_anything)\b"
        
    # Escape special regex characters and join with OR
    escaped_entities = [re.escape(entity.strip()) for entity in entity_list if entity.strip()]
    
    # Sort by length (longer first) to ensure longest matches
    escaped_entities = sorted(escaped_entities, key=len, reverse=True)
    
    # Create word boundary regex pattern
    pattern = r"\b(?:" + "|".join(escaped_entities) + r")\b"
    return pattern

# Define generate_regex_for_combinations function
def generate_regex_for_combinations(entity_list):
    """
    Generate a regex pattern for abbreviations and combinations
    
    Args:
        entity_list (List[str]): List of abbreviation strings
        
    Returns:
        str: Regex pattern matching any of the abbreviations
    """
    if not entity_list:
        return r"\b(?:placeholder_that_wont_match_anything)\b"
        
    # For abbreviations, we might want to match without word boundaries in some cases
    escaped_entities = [re.escape(entity.strip()) for entity in entity_list if entity.strip()]
    escaped_entities = sorted(escaped_entities, key=len, reverse=True)
    
    pattern = r"(?:" + "|".join(escaped_entities) + r")"
    return pattern

# Define AzureAILanguageRecognizer class
class AzureAILanguageRecognizer(EntityRecognizer):
    """
    A recognizer that uses Azure AI Language services for named entity recognition
    """
    
    def __init__(self, azure_ai_key, azure_ai_endpoint):
        super().__init__(supported_entities=["PERSON", "LOCATION", "ORGANIZATION"])
        self.azure_ai_key = azure_ai_key
        self.azure_ai_endpoint = azure_ai_endpoint
        self.name = "AzureAILanguageRecognizer"
        self.ignore_list = []
        
    def load(self) -> None:
        """Load the recognizer"""
        pass
        
    def analyze(self, text, entities=None, nlp_artifacts=None):
        """
        Analyze text using Azure AI Language
        In a real implementation, this would call the Azure AI Language API
        """
        # This is a placeholder implementation
        # In production, you would use Azure AI Language client to analyze text
        return []
        
class CustomNlpEngine(NlpEngine):
   
    def __init__(self):
        self.tokenizer_pattern = re.compile(r'\w+|[^\w\s]')
        self._supported_entities = []
        self._is_loaded = True
        
    def _simple_tokenize(self, text: str) -> List[Dict[str, Any]]:
        tokens = []
        matches = list(self.tokenizer_pattern.finditer(text))
        
        for i, match in enumerate(matches):
            start, end = match.span()
            token_text = match.group()
            
            token = {
                "text": token_text,
                "start": start,
                "end": end,
                "index": i,
                "is_space": token_text.isspace()
            }
            tokens.append(token)
            
        return tokens
    
    def _get_lemmas(self, tokens: List[Dict[str, Any]]) -> List[str]:
        return [token["text"].lower() for token in tokens]
    
    def create_nlp_artifacts(self, text: str) -> NlpArtifacts:
        tokens = self._simple_tokenize(text)
        lemmas = self._get_lemmas(tokens)

        return NlpArtifacts(
            tokens=tokens,
            lemmas=lemmas,
            entities=[], 
            tokens_indices=None,
            nlp_engine= "NlpEngine",
            language=None
        )

    def load(self) -> None:
        self._is_loaded = True
        
    def is_loaded(self) -> bool:
        return self._is_loaded
        
    def process_text(self, text: str, language: str) -> NlpArtifacts:
        return self.create_nlp_artifacts(text)
        
    def process_batch(self, texts: List[str], language: str) -> List[NlpArtifacts]:
        return [self.create_nlp_artifacts(text) for text in texts]
        
    def is_punct(self, word: str, language: str) -> bool:
        return all(not c.isalnum() for c in word)
        
    def get_supported_entities(self) -> List[str]:
        return self._supported_entities

class CustomTransformersRecognizer(EntityRecognizer):
    
    def __init__(self, model_path: str, supported_entities: List[str], ignore_list: Optional[List[str]] = None):
        super().__init__(supported_entities=supported_entities)
        self.model_path = model_path
        self.ignore_list = ignore_list or []
        self.name = "CustomTransformersRecognizer"
    
    def load_transformer(self, **kwargs):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        self.model = AutoModelForTokenClassification.from_pretrained(self.model_path)
        self.ner_pipeline = pipeline(
            "ner", 
            model=self.model, 
            tokenizer=self.tokenizer,
            aggregation_strategy="simple"
        )
        
    def analyze(self, text, entities=None, nlp_artifacts=None):
        """
        Analyze text using the transformer model
        
        Args:
            text (str): The text to analyze
            entities (List[str], optional): List of entities to look for
            nlp_artifacts (NlpArtifacts, optional): NLP artifacts
            
        Returns:
            List[RecognizerResult]: List of recognition results
        """
        # This is a placeholder method
        # In a real implementation, you would use the transformer model to analyze text
        return []

class NERTrainer:

    def __init__(self, model_name="bert-base-cased", output_dir="custom-ner-model"):
        self.model_name = model_name
        self.output_dir = output_dir
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.label_list = []
        self.label_to_id = {}
        self.id_to_label = {}
        
    def read_training_data(self, file_path):
        
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        entity_types = set()
        for example in data:
            for span in example.get('spans', []):
                entity_types.add(span['entity_type'])
        
        self.label_list = sorted(list(entity_types))
        self.label_list = ['O'] + [f"B-{label}" for label in self.label_list] + [f"I-{label}" for label in self.label_list]
        self.label_to_id = {label: i for i, label in enumerate(self.label_list)}
        self.id_to_label = {i: label for i, label in enumerate(self.label_list)}
        
        return data
    
    def preprocess_data(self, examples):
        processed_examples = []
        
        for example in examples:
            text = example['full_text']
            spans = example.get('spans', [])
            
            spans = sorted(spans, key=lambda x: x['start_position'])
            
            # Tokenize the text
            tokenized = self.tokenizer(
                text,
                truncation=True,
                padding="max_length",
                max_length=512,
                return_offsets_mapping=True,
                return_tensors="pt"
            )
            
            # Get offset mapping to convert character positions to token positions
            offset_mapping = tokenized.offset_mapping[0].tolist()
            
            # Initialize labels as "O" (outside any entity)
            labels = ["O"] * len(offset_mapping)
            
            # Map entity spans to token labels
            for span in spans:
                start_char = span['start_position']
                end_char = span['end_position']
                entity_type = span['entity_type']
                
                # Find the tokens that correspond to this entity
                token_start_idx = None
                token_end_idx = None
                
                for idx, (token_start, token_end) in enumerate(offset_mapping):
                    # Skip special tokens (like [CLS], [SEP])
                    if token_start == token_end == 0:
                        continue
                        
                    # Find the start token
                    if token_start_idx is None and token_start <= start_char < token_end:
                        token_start_idx = idx
                    
                    # Find the end token (inclusive)
                    if token_start <= end_char <= token_end:
                        token_end_idx = idx
                        break
                
                # Only label if we found valid token positions
                if token_start_idx is not None and token_end_idx is not None:
                    labels[token_start_idx] = f"B-{entity_type}"
                    for i in range(token_start_idx + 1, token_end_idx + 1):
                        labels[i] = f"I-{entity_type}"
            
            # Convert string labels to IDs
            label_ids = [self.label_to_id.get(label, 0) for label in labels]
            
            # Skip examples with no valid tokens
            if len(label_ids) == 0:
                continue
                
            # Create dataset entry
            processed_examples.append({
                "input_ids": tokenized["input_ids"][0],
                "attention_mask": tokenized["attention_mask"][0],
                "labels": torch.tensor(label_ids)
            })
        
        # Check if we have any valid examples
        if not processed_examples:
            raise ValueError("No valid training examples were generated after preprocessing. Check your data format.")
            
        return Dataset.from_list(processed_examples)
    
    def train_model(self, train_dataset, validation_dataset=None, epochs=3, batch_size=8):
        # Check if the dataset is empty
        if len(train_dataset) == 0:
            raise ValueError("Training dataset is empty. Cannot train the model.")
            
        # Verify label consistency
        if len(self.label_list) == 0:
            raise ValueError("No entity labels found. Check your training data.")
        
        model = AutoModelForTokenClassification.from_pretrained(
            self.model_name,
            num_labels=len(self.label_list),
            id2label=self.id_to_label,
            label2id=self.label_to_id
        )
        
        training_args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=2e-5,
            weight_decay=0.01,
            logging_dir="./logs",
            logging_steps=100,
            save_strategy="epoch"
        )
        
        data_collator = DataCollatorForTokenClassification(self.tokenizer)
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            data_collator=data_collator,
            tokenizer=self.tokenizer
        )
        
        trainer.train()
        trainer.save_model(self.output_dir)
        self.tokenizer.save_pretrained(self.output_dir)
        
        return model, self.tokenizer
        
    def save_label_mappings(self):
        with open(os.path.join(self.output_dir, "label_mappings.json"), "w") as f:
            json.dump({
                "label_list": self.label_list,
                "label_to_id": self.label_to_id,
                "id_to_label": {str(k): v for k, v in self.id_to_label.items()}  
            }, f)

def init_custom_ner(CONFIG=None, is_alm=True, company_list=[], company_abv=[], ticker_list=[], location_list=[], quotes_list=[], alm_creds=None, custom_ignore_list=None, model_path=None):
    
    if CONFIG is None:
        CONFIG = {
            "PRESIDIO_SUPPORTED_ENTITIES": [
                "PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "ORGANIZATION", 
                "URL", "LOCATION", "DATE_TIME"
            ]
        }
    
    custom_list = custom_ignore_list or []
    default_model_path = "FacebookAI/xlm-roberta-large-finetuned-conll03-english"
    
    selected_model_path = model_path if model_path else default_model_path
    
    supported_entities = CONFIG.get("PRESIDIO_SUPPORTED_ENTITIES")
    
    # Create a new empty registry
    registry = RecognizerRegistry()
    
    # Add recognizers one by one using try/except to catch any errors
    try:
        # Create and add the transformer recognizer
        transformers_recognizer = CustomTransformersRecognizer(
            model_path=selected_model_path,
            supported_entities=supported_entities,
            ignore_list=custom_list
        )
        transformers_recognizer.load_transformer()
        registry.add_recognizer(transformers_recognizer)
        logging.info("Added transformer recognizer")
    except Exception as e:
        logging.error(f"Failed to add transformer recognizer: {e}")

    # Add pattern recognizers one by one
    try:
        company_name_pattern = Pattern(name="company_name_pattern", regex=r'\b(?!(?:na|NA)\b)(?:[A-Z][a-z]*[ &-]?){1,4}(?<!\bInter)\b(?<!na\s)(?<!NA\s)(?:Ltd|ltd|LTD|llc|LLC|Inc\b(?![a-z])|inc\b(?![a-z])|INC\b(?![a-z])|plc|Corp|(?:Group\b(?!\s*[a-z]))|Aps|ApS|ASA|(?:Limited\b(?!\s*[a-z]))|Company|Co\b(?![a-z])|Corporation|LLP|GmbH|(?:AG\b(?!\s*[a-z]))|S\.A\.|Pty|Pte|LLP|(?:AS\b(?!\s*[a-z])))(?:,\s*Co\.|\.Ltd\.|\.LLC\.|\.Inc\.)?', score=0.8)
        company_recognizer = PatternRecognizer(supported_entity="ORGANIZATION", patterns=[company_name_pattern])
        registry.add_recognizer(company_recognizer)
        logging.info("Added company name recognizer")
    except Exception as e:
        logging.error(f"Failed to add company name recognizer: {e}")
    
    try:
        phone_num_pattern = Pattern(name="phone_num_pattern", regex=r'(?<!\d)\+\d{1,3}(?:[\s.-]|\s*\(\d+\)\s*)?(?:\d{1,4}[\s.-]?){2}(?:\d{1,4}[\s.-]?){1,2}(?![\d\s.-]*%)', score=1.0)
        phone_num_recognizer = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[phone_num_pattern])
        registry.add_recognizer(phone_num_recognizer)
        logging.info("Added phone number recognizer")
    except Exception as e:
        logging.error(f"Failed to add phone number recognizer: {e}")
    
    try:
        email_pattern = Pattern(name="email_pattern", regex=r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', score=0.8)
        email_recognizer = PatternRecognizer(supported_entity="EMAIL_ADDRESS", patterns=[email_pattern])
        registry.add_recognizer(email_recognizer)
        logging.info("Added email recognizer")
    except Exception as e:
        logging.error(f"Failed to add email recognizer: {e}")
    
    try:
        text_email_pattern = Pattern(name="text_email_pattern", regex=r'\b[A-Za-z0-9._%+-]+\(at\)[A-Za-z0-9.-]+\(dot\)[A-Z|a-z]{2,}\b', score=0.8)
        text_email_recognizer = PatternRecognizer(supported_entity="EMAIL_ADDRESS", patterns=[text_email_pattern])
        registry.add_recognizer(text_email_recognizer)
        logging.info("Added text email recognizer")
    except Exception as e:
        logging.error(f"Failed to add text email recognizer: {e}")

    try:
        project_name_pattern = Pattern(name="project_name_pattern", regex=r'\bProject\s[A-Z][a-z]*(?:\s[A-Z][a-z]*)*', score=0.8)
        project_name_recognizer = PatternRecognizer(supported_entity="PROJECT_NAME", patterns=[project_name_pattern])
        registry.add_recognizer(project_name_recognizer)
        logging.info("Added project name recognizer")
    except Exception as e:
        logging.error(f"Failed to add project name recognizer: {e}")
    
    try:
        word_phone_pattern = Pattern(name="word_phone_pattern", regex=r'1-[0-9]{3}-[A-Z]+', score=1.0)
        word_phone_recognizer = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[word_phone_pattern])
        registry.add_recognizer(word_phone_recognizer)
        logging.info("Added word phone recognizer")
    except Exception as e:
        logging.error(f"Failed to add word phone recognizer: {e}")
    
    # Add company list recognizer if list is not empty
    if company_list:
        try:
            regex_pattern = generate_regex(company_list)
            company_list_pattern = Pattern(name="company_list_pattern", regex=regex_pattern, score=0.8)
            company_list_recognizer = PatternRecognizer(supported_entity="ORGANIZATION", patterns=[company_list_pattern])
            registry.add_recognizer(company_list_recognizer)
            logging.info("Added company list recognizer")
        except Exception as e:
            logging.error(f"Failed to add company list recognizer: {e}")

    # Add company abbreviation recognizer if list is not empty
    if company_abv:
        try:
            regex_pattern_abv = generate_regex_for_combinations(company_abv)
            company_abv_pattern = Pattern(name="company_abv_pattern", regex=regex_pattern_abv, score=0.8)
            company_abv_recognizer = PatternRecognizer(supported_entity="ORGANIZATION", patterns=[company_abv_pattern])
            registry.add_recognizer(company_abv_recognizer)
            logging.info("Added company abbreviation recognizer")
        except Exception as e:
            logging.error(f"Failed to add company abbreviation recognizer: {e}")

    # Add ticker recognizer if list is not empty
    if ticker_list:
        try:
            regex_pattern_ticker = generate_regex(ticker_list)
            ticker_pattern = Pattern(name="ticker_pattern", regex=regex_pattern_ticker, score=0.8)
            ticker_recognizer = PatternRecognizer(supported_entity="ORGANIZATION", patterns=[ticker_pattern])
            registry.add_recognizer(ticker_recognizer)
            logging.info("Added ticker recognizer")
        except Exception as e:
            logging.error(f"Failed to add ticker recognizer: {e}")
    
    try:
        url_pattern = Pattern(name="url_pattern", regex=r'(?i)(http.{0,22}|www\..{0,22})', score=0.8)
        url_recognizer = PatternRecognizer(supported_entity="URL", patterns=[url_pattern])
        registry.add_recognizer(url_recognizer)
        logging.info("Added URL recognizer")
    except Exception as e:
        logging.error(f"Failed to add URL recognizer: {e}")

    # Add quotes list recognizer if list is not empty
    if quotes_list:
        try:
            regex_pattern_3 = generate_regex(quotes_list)
            quotes_list_pattern = Pattern(name="quotes_list_pattern", regex=regex_pattern_3, score=0.8)
            quotes_list_recognizer = PatternRecognizer(supported_entity="QUOTES", patterns=[quotes_list_pattern])
            registry.add_recognizer(quotes_list_recognizer)
            logging.info("Added quotes list recognizer")
        except Exception as e:
            logging.error(f"Failed to add quotes list recognizer: {e}")
    
    try:
        quotes_pattern = Pattern(name="quotes_pattern", regex=r'["\']([^"\']+)["\']', score=0.8)
        quotes_pattern_recognizer = PatternRecognizer(supported_entity="QUOTE", patterns=[quotes_pattern])
        registry.add_recognizer(quotes_pattern_recognizer)
        logging.info("Added quotes pattern recognizer")
    except Exception as e:
        logging.error(f"Failed to add quotes pattern recognizer: {e}")

    # Azure AI Language integration
    if is_alm and alm_creds:
        try:
            AZURE_AI_ENDPOINT = alm_creds.get('endpoint', 'https://uksdblang.cognitiveservices.azure.com')
            AZURE_AI_KEY = alm_creds.get('key', 'fd2e0f62222a4a1d99550b4a59dd4f57')
            
            azure_ai_language = AzureAILanguageRecognizer(
                azure_ai_key=AZURE_AI_KEY,
                azure_ai_endpoint=AZURE_AI_ENDPOINT
            )
            # Set ignore list separately if available
            if hasattr(azure_ai_language, 'ignore_list'):
                azure_ai_language.ignore_list = custom_list
            registry.add_recognizer(azure_ai_language)
            logging.info("Added Azure AI Language recognizer")
        except Exception as e:
            logging.warning(f"Could not initialize Azure AI Language: {e}")
    
    # Create the NLP engine and analyzer
    nlp_engine = CustomNlpEngine()
    
    # Verify recognizers before creating analyzer
    valid_recognizers = []
    for recognizer in registry.recognizers:
        if hasattr(recognizer, 'name') and recognizer.name != "SpacyRecognizer":
            # Double check it's an EntityRecognizer
            if isinstance(recognizer, EntityRecognizer):
                valid_recognizers.append(recognizer)
                logging.info(f"Valid recognizer: {recognizer.name}")
            else:
                logging.warning(f"Invalid recognizer type: {type(recognizer)}")
    
    # Create a new registry with only valid recognizers
    new_registry = RecognizerRegistry()
    new_registry.recognizers = valid_recognizers
    
    analyzer = AnalyzerEngine(
        registry=new_registry, 
        nlp_engine=nlp_engine, 
        supported_languages=["en"],
        default_score_threshold=0.7
    )

    return analyzer

def read_training_data(file_path):
    
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            return data
        except json.JSONDecodeError:
            data = []
            for line in f:
                try:
                    example = json.loads(line.strip())
                    data.append(example)
                except json.JSONDecodeError as e:
                    print(f"Error parsing line: {line}\nError: {e}")
    return data

def extract_entities_from_training_data(data):
    
    entities = {
        "PERSON": set(),
        "ORGANIZATION": set(), 
        "LOCATION": set(),
        "EMAIL": set(),
        "PHONE_NUMBER": set(),
        "COUNTRY": set(),
        "TITLE": set(),
        "QUOTE": set()
    }
    
    for example in data:
        for span in example.get('spans', []):
            entity_type = span['entity_type']
            entity_value = example['full_text'][span['start_position']:span['end_position']]  # Extract actual text
            
            if entity_type in entities:
                entities[entity_type].add(entity_value)
    
    return {k: list(v) for k, v in entities.items()}

def validate_data(data):
    """Validate training data format and integrity"""
    if not data:
        raise ValueError("Training data is empty")
        
    valid_count = 0
    for i, example in enumerate(data):
        try:
            # Check required fields
            if 'full_text' not in example:
                print(f"Warning: Example {i} missing 'full_text' field")
                continue
                
            text = example['full_text']
            
            if not isinstance(text, str) or not text:
                print(f"Warning: Example {i} has invalid text")
                continue
                
            # Check spans
            spans = example.get('spans', [])
            if not spans:
                print(f"Warning: Example {i} has no entity spans")
                
            valid_spans = True
            for span in spans:
                required_keys = ['entity_type', 'start_position', 'end_position']
                if not all(k in span for k in required_keys):
                    print(f"Warning: Span in example {i} missing required keys")
                    valid_spans = False
                    continue
                    
                # Validate span positions
                start = span['start_position']
                end = span['end_position']
                
                if not (isinstance(start, int) and isinstance(end, int)):
                    print(f"Warning: Non-integer span positions in example {i}")
                    valid_spans = False
                    continue
                    
                if not (0 <= start < end <= len(text)):
                    print(f"Warning: Invalid span positions in example {i}: start={start}, end={end}, text_len={len(text)}")
                    valid_spans = False
                    continue
            
            if valid_spans:
                valid_count += 1
                
        except Exception as e:
            print(f"Error validating example {i}: {e}")
    
    print(f"Found {valid_count} valid examples out of {len(data)}")
    if valid_count == 0:
        raise ValueError("No valid training examples found")
        
    return valid_count > 0

def main():

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    
    train_file = "dataset.txt"
    output_dir = "custom_ner_model"
    
    logger.info("Reading training data...")
    train_data = read_training_data(train_file)
    logger.info(f"Read {len(train_data)} training examples")
    
    # Validate data before processing
    logger.info("Validating training data...")
    if not validate_data(train_data):
        logger.error("Validation failed. Exiting.")
        return
    
    entity_lists = extract_entities_from_training_data(train_data)
    logger.info(f"Extracted entity lists: {', '.join(f'{k}: {len(v)}' for k, v in entity_lists.items())}")
    
    logger.info("Initializing NER trainer...")
    trainer = NERTrainer(model_name="bert-base-cased", output_dir=output_dir)
    
    # Set up label list before preprocessing
    entity_types = set()
    for example in train_data:
        for span in example.get('spans', []):
            entity_types.add(span['entity_type'])
    
    trainer.label_list = sorted(list(entity_types))
    if not trainer.label_list:
        logger.error("No entity types found in training data. Exiting.")
        return
        
    trainer.label_list = ['O'] + [f"B-{label}" for label in trainer.label_list] + [f"I-{label}" for label in trainer.label_list]
    trainer.label_to_id = {label: i for i, label in enumerate(trainer.label_list)}
    trainer.id_to_label = {i: label for i, label in enumerate(trainer.label_list)}
    
    logger.info(f"Using label list: {trainer.label_list}")
    
    try:
        logger.info("Preprocessing training data...")
        processed_data = trainer.preprocess_data(train_data)
        
        logger.info(f"Processed data contains {len(processed_data)} examples")
        
        logger.info("Training model...")
        model, tokenizer = trainer.train_model(processed_data, epochs=1)
        
        trainer.save_label_mappings()
        logger.info(f"Model trained and saved to {output_dir}")
        
    except Exception as e:
        logger.error(f"Error during training process: {e}")
        return
    
    logger.info("Initializing custom NER system with trained model...")
    try:
        analyzer = init_custom_ner(
            is_alm=False,
            company_list=entity_lists.get("ORGANIZATION", []),
            ticker_list=entity_lists.get("ORGANIZATION", []),
            location_list=entity_lists.get("LOCATION", []),\
            quotes_list=entity_lists.get("QUOTE", []),
            model_path=output_dir  # Use the trained model
        )
        
        test_text = "John Smith works at Microsoft and can be reached at john.smith@example.com or +1-555-123-4567."
        results = analyzer.analyze(text=test_text,language="en")
        
        logger.info("Test results:")
        for result in results:
            logger.info(f"Entity: {result.entity_type}, Text: {test_text[result.start:result.end]}, Score: {result.score}")
            
    except Exception as e:
        logger.error(f"Error initializing or testing NER system: {e}")

if __name__ == "__main__":
    main()

In [ ]:
!pip install --trusted-host github.com --trusted-host files.pythonhosted.org --trusted-host pypi.org https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.7.1/en_core_web_lg-3.7.1-py3-none-any.whl


In [12]:
import spacy
nlp = spacy.load("en_core_web_lg")

doc = nlp("Apple, Jet Blue and HSBC was founded by Steve Jobs and Margarine Unie, Lever Brothers respectively")

for ent in doc.ents:
    print(f"{ent.text} ({ent.label_})")

Apple (ORG)
Jet Blue (ORG)
HSBC (ORG)
Steve Jobs (PERSON)
Margarine Unie (PERSON)
Lever Brothers (ORG)


In [2]:
import spacy
import re
import pickle
import json
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os
from docx import Document
from openpyxl import load_workbook
from pptx import Presentation
import io
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class SpacyOrganizationNER:
    def __init__(self, model_name="en_core_web_sm", max_chunk_size=10000):
        
        self.nlp = spacy.load(model_name)
        self.max_chunk_size = max_chunk_size  
        self.target_entity_types = {'ORGANIZATION'}
        self.confidence_threshold = 0.7
        
        self.org_patterns = [
            r'\b(?:[A-Z][a-z]+(?:[ &-][A-Z][a-z]+)*\s)?(?:Ltd|LLC|Inc\.?|plc|Group|Limited|Company|Corporation|GmbH|AG|S\.A\.|Pty|Pte|Co\.?)\b',
            r'\b(?:[A-Z][a-z]+\s)+(?:Association|Institute|University|College|School|Foundation|Society|Agency|Authority|Commission|Department|Committee|Council|Bureau|Office)\b',
            r'\b[A-Z]{2,}(?:\s[A-Z]{2,}){0,3}\b(?!\s*\d+)',
        ]
        
        self.gazetteers = {'ORGANIZATION': set()}
        
        self.non_org_patterns = [
            r'\b(?:EUR|USD|GBP|JPY|CHF)[0-9,.]+[bm]?',
            r'\b[0-9,.]+(?:bp|bps|m|bn|k)\b',
            r'\b(?:Q[1-4]|[1-4]Q)\b',  
            r'\b(?:FY|H1|H2)\d{2,4}\b',  
            r'\b\d+(?:st|nd|rd|th)\b',  
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{4}\b',
            r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',
            r'\b(?:EPS|EBITDA|EBIT|PBT|ROE|ROA|FCF|P/E|EV|PB|DPS|YTD|YoY|MoM|TTM|LTM|NTM)\b',
            r'\b(?:FX|SKU|A&P|M&A|R&D|P&L|DCF|WACC|ESG|GHG)\b',
        ]
        
        self.company_semantic_references = [
            "company", "corporation", "firm", "enterprise", "business", 
            "organization", "incorporated", "inc", "ltd", "limited",
            "group", "holdings", "plc", "ag", "gmbh", "co",'corporate'
            # Added more semantic references
            "brand", "airline", "carrier", "bank", "retailer", "manufacturer",
            "provider", "vendor", "startup", "conglomerate", "multinational",
            "tech", "pharmaceutical", "automotive", "financial", "media"
        ]
        
        self.company_vectors = np.array([self.nlp(term).vector for term in self.company_semantic_references])
        
        self.ruler = self.nlp.add_pipe("entity_ruler", before="ner")
        self.update_entity_ruler()

    def update_entity_ruler(self):
        patterns = []
        for org in self.gazetteers['ORGANIZATION']:
            if len(org.strip()) > 2:
                patterns.append({"label": "ORG", "pattern": org.strip()})
        
        self.ruler.add_patterns(patterns)
    
    def load_entity_list(self, file_path):
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                
            if ',' in content:
                entities = [entity.strip() for entity in content.split(',') if entity.strip()]
            else:
                entities = [entity.strip() for entity in content.split('\n') if entity.strip()]
            
            entities = [entity for entity in entities if len(entity) >= 2]
            self.gazetteers['ORGANIZATION'].update(entities)
            
            self.update_entity_ruler()
            
            print(f"Loaded {len(entities)} organizations from {file_path}")
            return entities
        except Exception as e:
            print(f"Error loading entity list from {file_path}: {e}")
            return []
    
    def is_non_organization(self, text):
     
        cleaned_text = text.strip().strip('|').strip()
        
        if not cleaned_text:
            return True
            
        for pattern in self.non_org_patterns:
            if re.match(pattern, cleaned_text, re.IGNORECASE):
                return True
        
        if re.search(r'[A-Z]\d+', cleaned_text): 
            return True
        if "=" in cleaned_text or "IF(" in cleaned_text or "SUM(" in cleaned_text:
            return True
        if re.search(r'^\d+(?:[,.]\d+)*(?:[km]|bn|bp|bps)?$', cleaned_text):
            return True
        if re.match(r'^\d+$', cleaned_text) or re.match(r'^\d{1,2}/\d{1,2}$', cleaned_text):
            return True
        if len(cleaned_text) <= 2:
            return True
        if re.match(r'^[+\-]\d', cleaned_text):
            return True
        if '|' in cleaned_text: 
            return True
        if re.search(r'Sheet\d*!', cleaned_text): 
            return True
            
        return False
    
    def context_based_filtering(self, doc, entity_start, entity_end):
 
        entity_tokens = [token for token in doc if entity_start <= token.idx < entity_end]
        if not entity_tokens:
            return False
            
        context_window = 5 
        start_idx = max(0, entity_tokens[0].i - context_window)
        end_idx = min(len(doc), entity_tokens[-1].i + context_window + 1)
        
        financial_indicators = ['$', '%', '€', '£', 'million', 'billion', 'rate', 'growth', 'margin', 'price']
        financial_context = False
        
        for token in doc[start_idx:end_idx]:
            if token.text.lower() in financial_indicators or token.like_num:
                financial_context = True
                
        if financial_context and re.search(r'\d', ''.join([t.text for t in entity_tokens])):
            return False
            
        return True
    
    def semantic_company_similarity(self, text):
   
        if len(text.strip()) < 3:
            return 0
            
        text_vector = self.nlp(text).vector
        similarities = cosine_similarity([text_vector], self.company_vectors)[0]
        return np.max(similarities)
    
    def rule_based_extraction(self, text):
       
        entities = []
        doc = self.nlp(text)  
        
        for pattern in self.org_patterns:
            for match in re.finditer(pattern, text):
                start = match.start()
                end = match.end()
                entity_text = text[start:end].strip()
                
                if self.is_non_organization(entity_text):
                    continue
                    
                if len(entity_text) < 3:
                    continue
                    
                if not self.context_based_filtering(doc, start, end):
                    continue
                
                semantic_score = self.semantic_company_similarity(entity_text)
                
                confidence = 0.8 + 0.2 * semantic_score 
                    
                entities.append({
                    'entity_type': 'ORGANIZATION',
                    'start_position': start,
                    'end_position': end,
                    'text': entity_text,
                    'confidence': confidence,
                    'semantic_score': semantic_score
                })
        
        gazetteer = self.gazetteers['ORGANIZATION']
        if gazetteer:
            sorted_entities = sorted(gazetteer, key=len, reverse=True)
            
            for entity in sorted_entities:
                if len(entity.strip()) < 3:
                    continue
                    
                pattern = r'\b' + re.escape(entity) + r'\b'
                for match in re.finditer(pattern, text, re.IGNORECASE):
                    start = match.start()
                    end = match.end()
                    entity_text = text[start:end].strip()
                    
                    if self.is_non_organization(entity_text):
                        continue
                    
                    overlap = False
                    for existing in entities:
                        if (start < existing['end_position'] and end > existing['start_position']):
                            overlap = True
                            break
                    
                    if not overlap:
               
                        semantic_score = self.semantic_company_similarity(entity_text)
                        
                        confidence = 0.8 + 0.2 * semantic_score 
                        
                        entities.append({
                            'entity_type': 'ORGANIZATION',
                            'start_position': start,
                            'end_position': end,
                            'text': entity_text,
                            'confidence': confidence,
                            'semantic_score': semantic_score
                        })
        
        return entities
    
    def split_text_into_chunks(self, text):
        import re
       
        chunks = []
        chunk_size = self.max_chunk_size
        
        paragraphs = text.split('\n\n')
        current_chunk = ""
        
        for paragraph in paragraphs:
            if len(current_chunk) + len(paragraph) + 2 <= chunk_size:
                if current_chunk:
                    current_chunk += '\n\n' + paragraph
                else:
                    current_chunk = paragraph
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                
                if len(paragraph) > chunk_size:
                    sentences = re.split(r'(?<=[.!?])\s+', paragraph)
                    current_chunk = ""
                    
                    for sentence in sentences:
                        if len(current_chunk) + len(sentence) + 1 <= chunk_size:
                            if current_chunk:
                                current_chunk += ' ' + sentence
                            else:
                                current_chunk = sentence
                        else:
                            if current_chunk:
                                chunks.append(current_chunk)
                            
                            if len(sentence) > chunk_size:
                                for i in range(0, len(sentence), chunk_size):
                                    chunks.append(sentence[i:i+chunk_size])
                                current_chunk = ""
                            else:
                                current_chunk = sentence
                    
                    if current_chunk:
                        chunks.append(current_chunk)
                else:
                    current_chunk = paragraph
        
        if current_chunk:
            chunks.append(current_chunk)
            
        return chunks
    
    def extract_organizations(self, text):
        
        if len(text) > self.max_chunk_size:

            chunks = self.split_text_into_chunks(text)
            
            all_entities = []
            offset = 0
            
            for i, chunk in enumerate(chunks):
                
                chunk_entities = self.extract_organizations_from_chunk(chunk)
                
                for entity in chunk_entities:
                    entity['start_position'] += offset
                    entity['end_position'] += offset
                
                all_entities.extend(chunk_entities)
                offset += len(chunk)
            
            return all_entities
        else:
            return self.extract_organizations_from_chunk(text)
    
    def extract_organizations_from_chunk(self, text):
        
        doc = self.nlp(text)
        
        rule_based_entities = self.rule_based_extraction(text)
        
        spacy_entities = []
        for ent in doc.ents:
            if ent.label_ == "ORG":
                entity_text = ent.text
                
                if self.is_non_organization(entity_text):
                    continue
                
                if re.search(r'\d', entity_text) and len(entity_text) < 10:
                    if not any(re.search(pattern, entity_text) for pattern in self.org_patterns):
                        continue
                
                semantic_score = self.semantic_company_similarity(entity_text)
                
                confidence = 0.8 + 0.2 * semantic_score 
                
                spacy_entities.append({
                    'entity_type': 'ORGANIZATION',
                    'start_position': ent.start_char,
                    'end_position': ent.end_char,
                    'text': entity_text,
                    'confidence': confidence,
                    'semantic_score': semantic_score
                })
        
        combined_entities = rule_based_entities + spacy_entities
        combined_entities.sort(key=lambda x: x['start_position'])
        
        final_entities = []
        for entity in combined_entities:
            overlap_idx = -1
            for i, existing in enumerate(final_entities):
                if (entity['start_position'] < existing['end_position'] and 
                    entity['end_position'] > existing['start_position']):
                    overlap_idx = i
                    if entity.get('confidence', 0) > existing.get('confidence', 0):
                        final_entities[i] = entity
                    break
            
            if overlap_idx == -1:
                final_entities.append(entity)
        
        filtered_entities = []
        for entity in final_entities:
            entity_text = entity['text']
            
            if re.match(r'^\s*[€$£¥]\s*\d', entity_text):
                continue
                
            filtered_entities.append(entity)
        
        return filtered_entities
    
    def save_model(self, model_path='spacy_org_ner_model.pkl'):
        import pickle
  
        with open(model_path, 'wb') as f:
            pickle.dump({
                'gazetteers': self.gazetteers,
                'org_patterns': self.org_patterns,
                'non_org_patterns': self.non_org_patterns,
                'confidence_threshold': self.confidence_threshold,
                'company_semantic_references': self.company_semantic_references,
                'max_chunk_size': self.max_chunk_size
            }, f)
            
        print(f"Model saved to {model_path}")

    def load_model(self, model_path='spacy_org_ner_model.pkl'):
        import pickle
   
        try:
            with open(model_path, 'rb') as f:
                data = pickle.load(f)
                if 'gazetteers' in data:
                    self.gazetteers = data['gazetteers']
                if 'org_patterns' in data:
                    self.org_patterns = data['org_patterns']
                if 'non_org_patterns' in data:
                    self.non_org_patterns = data['non_org_patterns']
                if 'confidence_threshold' in data:
                    self.confidence_threshold = data['confidence_threshold']
                if 'company_semantic_references' in data:
                    self.company_semantic_references = data['company_semantic_references']
                    self.company_vectors = np.array([self.nlp(term).vector for term in self.company_semantic_references])
                if 'max_chunk_size' in data:
                    self.max_chunk_size = data['max_chunk_size']
            
            self.update_entity_ruler()
                    
            print(f"Model loaded from {model_path}")
            return True
        except Exception as e:
            print(f"Error loading model: {e}")
            return False

def extract_text_from_document(file_path):
    
    file_extension = os.path.splitext(file_path)[1].lower()
    extracted_text = ""
    
    try:
        if file_extension == '.pdf':
            with open(file_path, 'rb') as file:
                pdf = PdfReader(file)
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        extracted_text += page_text + "\n\n"
        
        elif file_extension == '.docx':
            doc = Document(file_path)
            paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
            extracted_text = "\n".join(paragraphs)
            
            for table in doc.tables:
                for row in table.rows:
                    row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                    if row_text:
                        extracted_text += "\n" + row_text
        
        elif file_extension == '.xlsx':
            wb = load_workbook(filename=file_path, read_only=True)
            sheet_texts = []
            
            for sheet in wb.worksheets:
                sheet_content = []
                for row in sheet.iter_rows(values_only=True):
                    row_values = [str(cell) if cell is not None else "" for cell in row]
                    if any(cell for cell in row_values):
                        sheet_content.append(" | ".join(row_values))
                
                if sheet_content:
                    sheet_texts.append(f"Sheet: {sheet.title}\n" + "\n".join(sheet_content))
            
            extracted_text = "\n\n".join(sheet_texts)
        
        elif file_extension == '.pptx':
            prs = Presentation(file_path)
            slides_text = []
            
            for i, slide in enumerate(prs.slides):
                slide_content = []
                slide_content.append(f"Slide {i+1}")
                
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text.strip():
                        slide_content.append(shape.text.strip())
                
                for shape in slide.shapes:
                    if shape.has_table:
                        for row in shape.table.rows:
                            row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                            if row_text:
                                slide_content.append(row_text)
                
                if len(slide_content) > 1:  
                    slides_text.append("\n".join(slide_content))
            
            extracted_text = "\n\n".join(slides_text)
        
        else:
            return f"Unsupported file format: {file_extension}"
        
        return extracted_text
    
    except Exception as e:
        return f"Error extracting text from {file_path}: {str(e)}"


def process_document_for_organizations(file_path, ner_model=None):
 
    if ner_model is None:
        ner_model = SpacyOrganizationNER()
    
    text = extract_text_from_document(file_path)
    
    entities = ner_model.extract_organizations(text)
    
    org_list = []
    for entity in entities:
        if 'text' in entity:  
            entity_text = entity['text']
        elif 'start_position' in entity and 'end_position' in entity:
     
            chunk_start = max(0, entity['start_position'] - 100)
            chunk_end = min(len(text), entity['end_position'] + 100)
            text_chunk = text[chunk_start:chunk_end]
            relative_start = entity['start_position'] - chunk_start
            relative_end = entity['end_position'] - chunk_start
            
            if 0 <= relative_start < relative_end <= len(text_chunk):
                entity_text = text_chunk[relative_start:relative_end]
            else:
                continue  
        else:
            continue
            
        if entity.get('confidence', 0) >= 0.75 and entity.get('semantic_score', 0) >= 0.2:
            entity_text = entity_text.strip().rstrip('|').lstrip('|').strip()
            if  entity_text:
                org_list.append(entity_text)
    
    clean_org_list = []
    for org in org_list:
        org = org.strip()
        
        if len(org) < 3:
            continue
        if re.match(r'^\d+$', org):
            continue
        if re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}$', org):
            continue
            
        clean_org_list.append(org)
    
    unique_orgs = sorted(list(set(clean_org_list)))
    return unique_orgs

if __name__ == "__main__":

    ner_model = SpacyOrganizationNER(max_chunk_size=10000)
    
    file_path = r"C:\Users\RW565TZ\Downloads\cmpny_test_POC.xlsx"
    
    if os.path.exists(file_path):
        organizations = process_document_for_organizations(file_path, ner_model)
        
        print(f"Found {len(organizations)} organizations in {file_path}:")
        for org in organizations:
            print(f"{org}")
    else:
        print(f"File not found: {file_path}")

Found 3 organizations in C:\Users\RW565TZ\Downloads\cmpny_test_POC.xlsx:
Amex India
EY GDS
LLP
